# 🏆 Day 3 Gauntlet — Clean the Titanic Dataset from Scratch

This gauntlet is a **complete real-world data cleaning pipeline** on the Titanic dataset.  
It applies every Day 3 topic in the correct order that you'd follow in a production data science project.

## The Full Pipeline

| Task | Operation | Topic |
|------|-----------|-------|
| 1 | Initial inspection — shape, info, missing counts | Topics 1 |
| 2 | Drop `Cabin` (77% missing); fill `Age` with median; fill `Embarked` with mode | Topics 2, 3 |
| 3 | Convert `Survived` → bool, `Pclass`/`Sex` → category | Topic 5 |
| 4 | Extract `Title` from `Name` using regex; group rare titles as 'Other' | Topics 6, 8 |
| 5 | Rename columns to snake_case | Topic 7 |
| 6 | Feature engineering: `family_size`, `is_alone`, `age_group` | Topics 9 |
| 7 | Map `Title` strings to numeric codes | Topic 8 |
| 8 | One-hot encode `embarked` with `drop_first=True` | Topic 10 |

## Why this order?
1. **Inspect first** — understand what you're dealing with
2. **Drop before fill** — no point filling a column you'll drop
3. **Fix types early** — ensures downstream operations behave correctly
4. **Engineer features** — create new columns from cleaned data
5. **Encode last** — dummy encoding expands the DataFrame width


Gauntlet #3: Clean the Titanic Dataset from Scratch


## Task 1 — Initial Inspection

Three essential first calls on any new dataset:
- `df.shape` → (891, 12) — 891 rows, 12 columns
- `df.info()` → shows dtype and non-null count per column
- `df.isna().sum()` → missing counts per column

**Findings:**
- `Age`: 177 missing (20%) — fill with median
- `Cabin`: 687 missing (77%) — drop the column
- `Embarked`: 2 missing (<1%) — fill with mode

`df.columns[df.isna().any()].tolist()` programmatically identifies all columns with any missing value.


In [1]:
import pandas as pd
import numpy as np

# Load fresh Titanic dataset
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# Task 1: Initial Inspection
print("=== Shape ===")
print(df.shape)

print("\n=== Info ===")
df.info()

print("\n=== Missing Values per Column ===")
print(df.isna().sum())

# Identify columns with missing values
missing_cols = df.columns[df.isna().any()].tolist()
print(f"\nColumns with missing values: {missing_cols}")

=== Shape ===
(891, 12)

=== Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

=== Missing Values per Column ===
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0


## Task 2, 3, 4 — Drop Cabin + Fill Age + Fill Embarked

Three imputation decisions in one cell:

**Drop `Cabin`:** 77% missing — no imputation strategy is reliable when 3 out of 4 values are unknown.

**Fill `Age` with median (28.0):** Median is preferred over mean because Age is slightly right-skewed. The median is robust to the few very old passengers.

**Fill `Embarked` with mode ('S'):** Mode is the only sensible fill for a categorical column. 'S' (Southampton) was the most common embarkation port. With only 2 missing values, this is a negligible imputation.


In [2]:
# Task 2: Drop Cabin column
df = df.drop('Cabin', axis=1)
print("Cabin dropped. Remaining columns:", df.columns.tolist())

# Task 3: Fill missing Age with median
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
print(f"Missing Age after fill: {df['Age'].isna().sum()}")

# Task 4: Fill missing Embarked with mode
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)
print(f"Missing Embarked after fill: {df['Embarked'].isna().sum()}")

Cabin dropped. Remaining columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']
Missing Age after fill: 0
Missing Embarked after fill: 0


## Task 5 — Data Type Conversions

Three semantic type corrections:

- `Survived → bool`: 0/1 integer → True/False. Makes the meaning explicit and enables `.mean()` as a survival rate.
- `Pclass → category`: 1, 2, 3 are nominal categories, not numbers. Saves memory and improves `.groupby()` performance.
- `Sex → category`: 'male'/'female' are nominal categories. Better than `object` for grouping operations.


In [3]:
# Task 5: Data type conversions
df['Survived'] = df['Survived'].astype(bool)
df['Pclass'] = df['Pclass'].astype('category')
df['Sex'] = df['Sex'].astype('category')

print("Updated dtypes:")
print(df.dtypes)

Updated dtypes:
PassengerId       int64
Survived           bool
Pclass         category
Name             object
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Embarked         object
dtype: object


## Task 6 — Extract Title from Name (Feature Engineering)

**Step 1 — Extract:** `r'([A-Za-z]+)\.'` — regex captures the word immediately before a period in each name.  
Example: `'Braund, Mr. Owen Harris'` → `'Mr'`

**Step 2 — Normalize:** `.str.lower()` → all lowercase so 'Mr' and 'mr' are treated the same.

**Step 3 — Group rare titles:** Titles like 'mlle', 'sir', 'countess', 'jonkheer' appear only 1–3 times.  
Replacing them with `'Other'` prevents overfitting to rare categories in ML models.

**Result:** 5 title groups — `mr`, `miss`, `mrs`, `master`, `Other`.


In [4]:
# Task 6: Extract Title from Name
df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.')
df['Title'] = df['Title'].str.lower()

# Define rare titles to be grouped as 'Other'
rare_titles = ['mlle', 'ms', 'mme', 'lady', 'sir', 'countess', 
               'jonkheer', 'don', 'rev', 'dr', 'major', 'col', 'capt']

df['Title'] = df['Title'].replace(rare_titles, 'Other')

print("Title value counts after cleaning:")
print(df['Title'].value_counts())

Title value counts after cleaning:
Title
mr        517
miss      182
mrs       125
master     40
Other      27
Name: count, dtype: int64


## Task 7 — Rename Columns to snake_case

Standardizes all column names to lowercase with underscores — the Python convention.  
`rename(columns={...})` maps old names to new names selectively.  
After this, columns like `siblings_spouses` are easier to type and less error-prone than `SibSp`.


In [5]:
# Task 7: Rename columns
df = df.rename(columns={
    'PassengerId': 'passenger_id',
    'Pclass': 'pclass',
    'SibSp': 'siblings_spouses',
    'Parch': 'parents_children',
    'Fare': 'fare',
    'Embarked': 'embarked'
})

print("Renamed columns:", df.columns.tolist())

Renamed columns: ['passenger_id', 'Survived', 'pclass', 'Name', 'Sex', 'Age', 'siblings_spouses', 'parents_children', 'Ticket', 'fare', 'embarked', 'Title']


## Task 8 — Feature Engineering: family_size, is_alone, age_group

Three new features from existing data:

**`family_size`:** `siblings_spouses + parents_children + 1` — total people in the passenger's group including themselves.

**`is_alone`:** `family_size == 1` → boolean flag for solo travelers.  
Research on the Titanic shows solo travelers had different survival rates than family groups.

**`age_group`:** `pd.cut()` with domain-meaningful bins:
- Child: 0–12, Teen: 13–18, Young Adult: 19–35, Adult: 36–60, Senior: 61+


In [6]:
# Task 8: family_size, is_alone, age_group
df['family_size'] = df['siblings_spouses'] + df['parents_children'] + 1
df['is_alone'] = df['family_size'] == 1

bins = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
df['age_group'] = pd.cut(df['Age'], bins=bins, labels=labels)

print(df[['family_size', 'is_alone', 'age_group']].head())

   family_size  is_alone    age_group
0            2     False  Young Adult
1            2     False        Adult
2            1      True  Young Adult
3            2     False  Young Adult
4            1      True  Young Adult


## Task 9 — Map Title Strings to Numeric Codes

`df['Title'].map({'mr': 0, 'mrs': 1, 'miss': 2, 'master': 3, 'other': 4})` converts the string titles to integers.  
Note: `.map()` sets unmatched values to NaN — check with `.isna().sum()` after mapping.  
This is necessary because most ML algorithms require numeric input — string labels won't work.

> **Alternative:** `pd.get_dummies(df['Title'])` would create one-hot encoded columns instead. Mapping preserves ordinal structure (though Title is really nominal here).


In [ ]:
# Task 9: Map Title strings to numeric codes
title_mapping = {
    'mr': 0,
    'mrs': 1,
    'miss': 2,
    'master': 3,
    'other': 4
}
df['Title'] = df['Title'].map(title_mapping)

print("Title numeric codes:")
print(df['Title'].value_counts().sort_index())

## Task 10 — One-Hot Encode Embarked

`pd.get_dummies(df['embarked'], prefix='emb', drop_first=True)`:
- `prefix='emb'` → new columns named `emb_Q` and `emb_S`
- `drop_first=True` → drops `emb_C` (Cherbourg); it's implied when both `emb_Q=0` and `emb_S=0`

`pd.concat([df, embarked_dummies], axis=1)` — adds the dummy columns horizontally.  
Then `df.drop('embarked', axis=1)` removes the original string column.

**Final shape: (891, 16)** — started with 12 columns, dropped Cabin (−1), dropped embarked (−1), added 5 features (+5: Title, family_size, is_alone, age_group, emb_Q, emb_S) = 16.


In [7]:
# Task 10: Dummies for embarked, drop first, and concatenate
embarked_dummies = pd.get_dummies(df['embarked'], prefix='emb', drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)

# Drop the original embarked column
df = df.drop('embarked', axis=1)

print("Final DataFrame shape:", df.shape)
print("Final columns:", df.columns.tolist())

Final DataFrame shape: (891, 16)
Final columns: ['passenger_id', 'Survived', 'pclass', 'Name', 'Sex', 'Age', 'siblings_spouses', 'parents_children', 'Ticket', 'fare', 'Title', 'family_size', 'is_alone', 'age_group', 'emb_Q', 'emb_S']
